In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}



def export_figure(figure, figure_id, tight=False):
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
data=load_npz("data/FigS8/selected_plot_data.npz")

In [ ]:
limits=data['common_y_limits'];x=data['updates']
assert np.array_equal(x,np.arange(501))
plt.rcParams.update({'font.family':'serif','font.serif':['DejaVu Serif'],'font.size':11,
   'axes.titlesize':12,'axes.labelsize':12,'xtick.labelsize':10,'ytick.labelsize':10,
   'legend.fontsize':10.5,'pdf.fonttype':42,'ps.fonttype':42,'axes.linewidth':.8,
   'figure.facecolor':'white','axes.facecolor':'white','axes.spines.top':True,'axes.spines.right':True})
fig=plt.figure(figsize=(11.4,4.05))
grid=fig.add_gridspec(1,3,left=.085,right=.985,bottom=.23,top=.78,wspace=.28)
labels=[r'Low: $[0.2,0.6]$',r'High: $(0.6,1]$']
colors=['#2468A2','#D17A12'];styles=['-','--'];frozen={};rows=[];checks=[]
for i,q in enumerate([4,8,16]):
    key=f'q{q}_interleaved';E=data[key+'_absolute'];R=data[key+'_ratio'];agg=data[key+'_aggregate']
    assert E.shape==(3,501,2) and np.array_equal(E/E[:,:1,:],R)
    assert np.array_equal(np.sort(R,axis=0),agg)
    for field in ['absolute','ratio','aggregate']:frozen[key+'_'+field]=data[key+'_'+field]
    ax=fig.add_subplot(grid[0,i]);ax.set_title(f'{q} qubits',pad=10)
    ax.text(-.14,1.08,f'({chr(97+i)})',transform=ax.transAxes,fontsize=12,fontweight='bold')
    ax.set_xlim(0,500);ax.set_xticks([0,100,200,300,400,500]);ax.set_yscale('log');ax.set_ylim(limits)
    ax.tick_params(direction='out',length=3.5,width=.8);ax.grid(alpha=.14,which='major',linewidth=.5)
    handles=[]
    for b in range(2):
        lo,med,hi=agg[:,:,b]
        ax.fill_between(x,lo,hi,color=colors[b],alpha=.08,linewidth=0)
        line,=ax.plot(x,med,color=colors[b],ls=styles[b],lw=1.25,label=labels[b])
        assert np.array_equal(line.get_ydata(),data[key+'_aggregate'][1,:,b])
        assert lo.min()>=limits[0] and hi.max()<=limits[1]
        handles.append(line)
        checks.append({'source_panel':'def'[i],'new_panel':'abc'[i],'qubits':q,'band':['low','high'][b],
                       'points':501,'line_and_minmax_identical_to_source':True})
        for t in range(501):rows.append({'new_panel':'abc'[i],'source_panel':'def'[i],'qubits':q,
            'band':['low','high'][b],'update':t,'minimum':float(lo[t]),'median':float(med[t]),'maximum':float(hi[t])})
    ax.axhline(1,color='.4',ls='--',lw=.65,zorder=0);ax.axhline(.5,color='.55',ls=':',lw=.65,zorder=0)
    if i==0:ax.set_ylabel(r'Relative radial error $R_G(n)$')
fig.legend(handles,labels,loc='center',bbox_to_anchor=(.53,.95),ncol=2,frameon=False,
           handlelength=2.5,columnspacing=1.6)
fig.supxlabel('Training iteration',fontsize=12,y=.045)

export_figure(fig,"FigS8",tight=True)
plt.close(fig)